# DataScout quickstart

Collects reports of COVID-19 contact-tracing app download counts, one stage at a time. Each stage's output can be reviewed and edited before the next one runs.

You need API keys in your environment: `OPENAI_API_KEY` (or another litellm provider), `BRIGHT_DATA_API_KEY`, `BRIGHT_DATA_ZONE` and `JINA_API_KEY`. Run `crawl4ai-setup` once to install the browser used for extraction. Running this notebook makes paid API calls.

In [ ]:
import logging
import os

from datascout import (
    BrightDataConfig,
    JinaConfig,
    LLMConfig,
    control_quality,
    extract_data,
    generate_extraction_schema,
    generate_search_queries,
    get_url_date,
    rerank_results_jina_api,
    retrieve_webpages,
)

logging.basicConfig(level=logging.WARNING, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logging.getLogger("datascout").setLevel(logging.INFO)

In [ ]:
llm_config = LLMConfig(api_key=os.environ.get("OPENAI_API_KEY"), model="openai/gpt-4.1", temperature=0.2)
bright_data_config = BrightDataConfig(api_key=os.environ["BRIGHT_DATA_API_KEY"], zone=os.environ["BRIGHT_DATA_ZONE"])
jina_config = JinaConfig(api_key=os.environ["JINA_API_KEY"])

In [ ]:
dataset_description = "Reports of download counts of COVID contact tracing apps in different U.S. states."
variables = {"state_and_app_name": ["Alabama GuideSafe"]}
# variables = {
#     "state_and_app_name": [
#         "Alabama GuideSafe",
#         "Arizona Covid Watch",
#         "California Covid Notify",
#         "Colorado Exposure Notifications",
#         "Connecticut Covid Alert",
#     ]
# }
variable_names = list(variables.keys())

# Search Query Generation

In [ ]:
num_queries = 1

query_templates = generate_search_queries(dataset_description, num_queries, llm_config, variable_names)
for query_template in query_templates:
    print(query_template)

# Web Page Retrieval

In [ ]:
results_pages_per_query = 1
start_date = "2020-01-01"
end_date = "2022-12-31"

webpages = retrieve_webpages(query_templates, results_pages_per_query, bright_data_config, variables)
# webpages = retrieve_webpages(
#     query_templates, results_pages_per_query, bright_data_config, variables, start_date, end_date, time_chunking=True
# )
# webpages = retrieve_webpages(query_templates, results_pages_per_query, bright_data_config, variables, news_only=True)
# webpages = retrieve_webpages(
#     query_templates,
#     results_pages_per_query,
#     bright_data_config,
#     variables,
#     media_cloud_country="United States",
#     geolocation_country="United States",
# )

# webpage_generator = retrieve_webpages_streaming(
#     query_templates, results_pages_per_query, bright_data_config, variables
# )
# for webpage in webpage_generator:
#     print(webpage)

In [ ]:
for w in webpages:
    print(w)
    for q in webpages[w]:
        print(q, len(webpages[w][q]))

In [ ]:
all_webpages = []
for w in webpages:
    for q in webpages[w]:
        for page in webpages[w][q]:
            page["query"] = q
            all_webpages.append(page)

all_webpages

In [ ]:
len(all_webpages)

In [ ]:
reranking_scores = rerank_results_jina_api(
    [p["query"] for p in all_webpages], [p["title"] for p in all_webpages], jina_config
)
reranking_scores[:3]

In [ ]:
score = {(r["query"], r["text"]): r["score"] for r in reranking_scores}

In [ ]:
for page in all_webpages:
    page["score"] = score[(page["query"], page["title"])]

In [ ]:
webpages_above_threshold = [p for p in all_webpages if p["score"] > 0.4]
webpages_above_threshold = sorted(webpages_above_threshold, key=lambda x: x["score"], reverse=True)
print(len(webpages_above_threshold))

webpages_above_threshold

In [ ]:
# Publication dates help when a page reports a count without a date.
get_url_date(webpages_above_threshold[0]["link"])

# Data Extraction

In [ ]:
schema_fields = ["state", "date"]
schema = generate_extraction_schema(dataset_description, schema_fields, llm_config)

print(schema)

In [ ]:
extracted_data = await extract_data([w["link"] for w in webpages_above_threshold], schema, llm_config)

In [ ]:
extracted_data

# Quality Control

In [ ]:
issues = control_quality(extracted_data, dataset_description, llm_config)

In [ ]:
issues